In [ ]:
import pandas as pd
import logging 
import os
import numpy as np

import src.constants as C
from src.preprocessing import process_store_data, drop_closed, drop_null_targets
from src.features import attach_store_data, make_features, make_targets
from src.engine import nested_cv

logging.basicConfig(
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(C.LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler(),
    ],
    force=True,  # ensure no duplicate handlers if this cell is re-run in a notebookcls
)

logger = logging.getLogger(__name__)

os.makedirs(C.LOG_DIR, exist_ok=True)

cv_config = {
    "n_outer_splits":   C.N_OUTER_SPLITS,
    "n_inner_splits":   C.N_INNER_SPLITS,
    "forecast_horizon": C.FORECAST_HORIZON,
    "outer_train_size": C.OUTER_TRAIN_SIZE,
    "inner_train_size": C.INNER_TRAIN_SIZE,
}

study_config = {
    "storage_url":           C.STORAGE_URL,
    "n_trials":              C.NUM_TRIALS,
    "n_startup_trials":      C.NUM_STARTUP_TRIALS,
    "n_jobs":                C.NUM_JOBS,
    "seed":                  C.SEED,
    "xgb_constants":         C.XGB_CONSTANTS,
    "early_stopping_rounds": C.EARLY_STOPPING_ROUNDS,
    "num_boost_rounds":      C.NUM_BOOST_ROUNDS,
    "monitor_periods":       C.MONITOR_PERIODS,
    "hyperparameters":       C.HYPERPARAMETERS,
    "log_dir":               C.LOG_DIR,
}


#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

c:\Users\m_kal\anaconda3\envs\rossmann\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
stores = pd.read_csv(C.STORE_FILE)
stores = process_store_data(stores)

df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_21300\2633813907.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [4]:

stores = pd.read_csv(C.STORE_FILE)
stores = process_store_data(stores)

df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)

df['Sales'] = df['Sales'].apply(np.log1p)
df = attach_store_data(df, stores).sort_values(['Store', 'Date'])
X = make_features(df, C.LAGS, C.ROLL_WINDOWS, C.DIFFS)
y = make_targets(df[['Date', 'Store', 'Sales']], C.FORECAST_HORIZON)
X, y = drop_closed(X, y)
X, y = drop_null_targets(X, y)

logger.info(f"Transformed dataset: {len(X)} samples, {X.shape[1]} features")

C:\Users\m_kal\AppData\Local\Temp\ipykernel_21300\921971754.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(C.TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
INFO:__main__:Transformed dataset: 798160 samples, 77 features


In [5]:
pd.concat([
    X.dtypes,
    X.isna().sum()/len(X),
    X.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('./artifacts/feature_summary.csv')

In [ ]:
#==================================================================================
num_stores = 5
X_small = X.xs(slice(None, num_stores), level="Store", drop_level=False)
y_small = y.xs(slice(None, num_stores), level="Store", drop_level=False)
#==================================================================================

nested_cv(X_small, y_small, cv_config, study_config)


INFO:src.engine.hypertuning:Outer fold 1/6  (train=2477 samples, test=206 samples)
INFO:src.engine.hypertuning:Running hyperparameter tuning (50 trials, 4 inner folds)...
[I 2026-05-29 14:37:34,436] A new study created in RDB with name: study_fold_1


[0]	train-rmse:3.04815+0.05317	test-rmse:3.10542+0.23048
[0]	train-rmse:3.28147+0.04553	test-rmse:3.25115+0.18776
[11]	train-rmse:1.75363+0.08466	test-rmse:3.32819+0.24401
[0]	train-rmse:3.26998+0.04555	test-rmse:3.25068+0.18777


[I 2026-05-29 14:37:38,238] Trial 2 finished with value: 3.044371004917 and parameters: {'max_depth': 2, 'learning_rate': 0.2350108420704842, 'subsample': 0.5250317403844522, 'colsample_bytree': 0.8631255512788409, 'min_child_weight': 0.37695294621768577, 'reg_alpha': 0.33822191650499817, 'reg_lambda': 8.693959327330038e-05, 'gamma': 4.151861034006719}. Best is trial 2 with value: 3.044371004917.


[0]	train-rmse:3.17992+0.04569	test-rmse:3.25010+0.18876
[0]	train-rmse:3.17540+0.04626	test-rmse:3.24884+0.18539
[100]	train-rmse:3.07760+0.05399	test-rmse:3.22992+0.19283
